# Chapter 6 – Searching with Quantum  
*From checking candidates to amplifying answers*

This notebook explores how quantum computing changes the way we think about **search**. We begin with a real dataset, expand it into a large search space, and define a small number of **gold patients** hidden among billions of possible patient-signature candidates.

We start by building the NHANES-inspired search space and running a classical search one candidate at a time. This establishes the baseline: when there is no structure to exploit, progress comes from repeated calls to an **oracle** that answers yes or no for each candidate.

From there, we model Grover’s algorithm directly with **amplitudes**. A small visualization window lets us see the core pattern clearly: encode the search space, mark the gold patient with a phase change, amplify that mark through diffusion, and repeat the process until the target becomes likely to appear.

The notebook then translates that same idea into a **quantum circuit**. Using Qiskit, we build a four-qubit Grover circuit, run it on a simulator, and observe how measurement results concentrate around the marked state.

The final example steps back to estimate scale. It compares classical search with Grover search across increasingly large spaces and shows how many index qubits and Grover iterations would be required for the full NHANES-inspired problem.

Each section follows the same pattern used throughout this chapter: `encode`, `mark`, `amplify`, and `refine`. As you run the code, focus on how the search space is represented, how the oracle creates a distinction, and how interference turns that distinction into a measurable advantage.

### Code 6-0: Preparing the NHANES Search Space

Before we can search for anything, we need a search space large enough to make the problem interesting. This example loads a subset of the NHANES dataset and expands each patient into more than one million possible genetic signatures, creating a search space containing over eight billion candidate patient-signature pairs.

The code then identifies three patients whose health profiles satisfy a predefined condition and plants them deep within the expanded search space. These become the gold patients used throughout the chapter.

Finally, a small sixteen-candidate visualization window is created around the first gold patient. This window allows us to explore the mechanics of Grover's algorithm using a manageable example while preserving a connection to the much larger search problem underneath.

The goal is to create a large, unstructured search space that lets us compare classical search with quantum search and explore how Grover's algorithm behaves as the number of candidates grows.

In [ ]:
import numpy as np
import pandas as pd

# === Code 6-0: Preparing the NHANES Search Space ===

url = "https://raw.githubusercontent.com/JerryCuomo/ThinkQuantum/main/datasets/nhanes_2026.csv"
cols = ["ID", "Age", "Gender", "BMI", "BPSysAve", "BPDiaAve", "TotChol", "Diabetes"]
df = pd.read_csv(url)[cols].dropna().reset_index(drop=True)

signatures_per_patient = 2**20
search_size = len(df) * signatures_per_patient

# Decode one search position into a patient and signature pair.
def decode_candidate(candidate):
    patient_index = candidate // signatures_per_patient
    signature_index = candidate % signatures_per_patient
    return patient_index, signature_index

# Identify gold patients whose health profile matches the condition.
gold_patients = df[(df.Age >= 50) & (df.BMI >= 30) & (df.BPSysAve >= 130)]
target_patients = gold_patients.sample(3, random_state=7).index.to_list()
target_signatures = [2**20 - 17, 2**19 + 509, 2**18 + 887]

# Plant the gold patients deep in the expanded search space.
needles = sorted([p * signatures_per_patient + s for p, s in zip(target_patients, target_signatures)])

# Create a small 16-candidate window so we can visualize one Grover iteration.
window_size = 16
window_target = 10
window_start = needles[0] - window_target
window_candidates = list(range(window_start, window_start + window_size))
assert window_candidates[window_target] == needles[0]

patient_index, signature_index = decode_candidate(needles[0])
classical_budget = 1_000_000_000

print(f"NHANES patients:        {len(df):,}")
print(f"Signatures per patient: {signatures_per_patient:,}")
print(f"Search candidates:      {search_size:,}")
print(f"Gold patients planted:  {len(needles)}")
print(f"First gold patient:     {needles[0]:,}")
print(f"Classical budget:       {classical_budget:,}")
print(f"Reachable classically:  {needles[0] < classical_budget}")

print("\nFirst gold patient details:")
print(f"Patient index:          {patient_index:,}")
print(f"Signature index:        {signature_index:,}")
print(df.loc[patient_index])

print("\nGrover visualization window:")
print(f"Window size:            {window_size}")
print(f"Window start:           {window_start:,}")
print(f"Gold patient position:  {window_target}")

### Code 6-1: Searching NHANES One Record at a Time

Before exploring quantum search, it helps to see why the problem is difficult classically. This example performs a straightforward brute-force search across the expanded `NHANES` search space created in Code 6-0. Each candidate is evaluated independently by an oracle that determines whether the candidate corresponds to one of the planted gold patients.

The search proceeds one candidate at a time until either a match is found or a predefined search budget is exhausted. Along the way, the code measures how many candidates were examined, how long the search required, and how far the search progressed relative to the location of the first gold patient.

The goal is to establish a baseline. By observing how quickly a classical search becomes overwhelmed by a large unstructured search space, we create a point of comparison for the quantum techniques that follow.

In [ ]:
import time

# === Code 6-1: Searching One Candidate at a Time ===

# Uses search_size, needles, classical_budget,
# decode_candidate(), and df from Code 6-0.

# Test whether a candidate corresponds to a gold patient.
def search_oracle(candidate):
    return candidate in needles

# Examine candidates until a gold patient is found or the budget is exhausted.
def classical_search(search_size, oracle, budget):
    start_time = time.perf_counter()

    for checks, candidate in enumerate(range(search_size), start=1):
        if checks > budget:
            elapsed = time.perf_counter() - start_time
            return checks - 1, None, elapsed
        if oracle(candidate):
            elapsed = time.perf_counter() - start_time
            return checks, candidate, elapsed

    elapsed = time.perf_counter() - start_time
    return checks, None, elapsed

checks, candidate, elapsed = classical_search(
    search_size,
    search_oracle,
    classical_budget
)

print(f"Search candidates:   {search_size:,}")
print(f"Search budget:       {classical_budget:,}")
print(f"Candidates checked:  {checks:,}")
print(f"Elapsed time:        {elapsed:.2f} seconds")
print(f"Search rate:         {checks / elapsed:,.0f} candidates/sec")
print(f"First gold patient:  {needles[0]:,}")
print(f"Gap to first target: {needles[0] - checks:,} candidates short")

if candidate is None:
    print("\nNo gold patient found within the search budget.")
else:
    patient_index, signature_index = decode_candidate(candidate)

    print("\nGold patient found:")
    print(f"Candidate index: {candidate:,}")
    print(f"Patient index:   {patient_index:,}")
    print(f"Signature index: {signature_index:,}")

    print("\nPatient details:")
    print(df.loc[patient_index])

### Code 6-2: Modeling a Grover Iteration with Amplitudes

This example models the core idea behind Grover's algorithm using a small sixteen-candidate search space. Each candidate is represented by an amplitude rather than a probability, allowing us to observe how the quantum state changes as the search progresses.

The code walks through the three key stages of a Grover iteration. First, all candidates begin with equal amplitude. Next, the oracle marks the gold patient by flipping the sign of its amplitude. Finally, the diffusion step reflects amplitudes around their average, increasing the amplitude of the marked state while slightly reducing the others.

The resulting plots provide a visual view of amplitude amplification. Although the changes may appear small after a single iteration, they establish the mechanism that allows Grover's algorithm to gradually increase the likelihood of measuring a desired outcome.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === Code 6-2: Performing One Grover Iteration ===
# Requires window_size and window_target from Code 6-0.

# Step 1: Encode all candidates in the window with equal amplitudes.
def encode_search_window(size):
    return np.ones(size) / np.sqrt(size)

# Step 2: Mark the gold patient by flipping its phase.
def apply_oracle(amplitudes, target):
    marked = amplitudes.copy()
    marked[target] *= -1
    return marked

# Step 3: Amplify the mark by reflecting around the average.
def apply_diffusion(amplitudes):
    mean = np.mean(amplitudes)
    return 2 * mean - amplitudes

# Apply one complete Grover iteration.
def grover_iteration(amplitudes, target):
    return apply_diffusion(apply_oracle(amplitudes, target))

# Visualize amplitude states.
def plot_states(states, target, titles):
    fig, axes = plt.subplots(1, len(states), figsize=(10.6, 6))

    for ax, amplitudes, title in zip(axes, states, titles):
        colors = ["silver"] * len(amplitudes)
        colors[target] = "gold"

        bars = ax.bar(range(len(amplitudes)), amplitudes, color=colors, edgecolor="white")
        bars[target].set_hatch("///")

        ax.axhline(np.mean(amplitudes), linestyle="--", linewidth=1.5)
        ax.set_title(title, fontsize=12)
        ax.set_xticks([0, 5, 10, 15])
        ax.set_ylim(-0.35, 0.75)

    axes[0].set_ylabel("Amplitude")
    plt.tight_layout()
    plt.show()

# Perform one Grover iteration
initial = encode_search_window(window_size)
marked = apply_oracle(initial, window_target)
amplified = apply_diffusion(marked)

plot_states(
    [initial, marked, amplified],
    window_target,
    ["Step 1: Encode", "Step 2: Mark", "Step 3: Amplify"]
)

print(f"Window size:                  {window_size}")
print(f"Gold patient position:        {window_target}")
print(f"Initial target probability:   {initial[window_target]**2:.4f}")
print(f"After one Grover iteration:   {amplified[window_target]**2:.4f}")

### Code 6-3: Refining the Search

A single Grover iteration improves the likelihood of measuring the target, but the process does not stop there. This example applies Grover's amplification step repeatedly and tracks how the target probability evolves over time.

With each iteration, probability flows toward the marked state and away from the surrounding candidates. The improvement is not indefinite, however. After reaching a peak, additional iterations begin to move probability away from the target, reducing the likelihood of a successful measurement.

The resulting plot illustrates an important feature of Grover's algorithm. Success depends not only on marking and amplifying the target, but also on stopping at the right moment. The number of iterations becomes part of the solution.


In [ ]:
# === Code 6-3: Refining Grover Iterations ===
# Requires initial, window_target, and grover_iteration() from Code 6-2.

max_iterations = 8
states = [initial]
current = initial

for _ in range(max_iterations):
    current = grover_iteration(current, window_target)
    states.append(current)

probs = [s[window_target]**2 for s in states]
best = int(np.argmax(probs))

colors = ["gold" if i == best else "silver"
          for i in range(len(probs))]

plt.figure(figsize=(10.6, 6))

bars = plt.bar(
    range(len(probs)),
    probs,
    color=colors,
    width=0.8,
    edgecolor="0.7",
    linewidth=0.4
)

bars[best].set_hatch("///")
bars[best].set_edgecolor("white")
bars[best].set_linewidth(0.2)

plt.plot(range(len(probs)), probs, marker="o", linewidth=2)
plt.axvline(best, linestyle="--", linewidth=1.5)

plt.xlabel("Grover Iteration")
plt.ylabel("Gold Patient Probability")
plt.xticks(range(len(probs)))
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

print(f"Best iteration: {best}")
print(f"Maximum gold patient probability: {probs[best]:.4f}")

### Code 6-4: Building a Grover Search Circuit

The previous examples modeled Grover's algorithm using amplitudes directly. This example takes the next step and implements the same search process as an actual quantum circuit using Qiskit.

The circuit begins by placing four qubits into equal superposition, creating a search space containing sixteen possible states. An oracle then marks the target state through a phase change, and a diffusion operator amplifies that mark by increasing the probability of measuring the desired outcome. Finally, the circuit is executed on a simulator and the measurement results are collected and visualized.

This example connects the conceptual model developed throughout the chapter with the gate-level operations used by a quantum computer. The resulting circuit demonstrates how the Encode, Mark, Amplify, and Refine pattern can be expressed using quantum gates and measurements.

**Note:** This example requires Qiskit and Aer. Execute the package installation cell immediately below this markdown cell before running the code.

In [ ]:
# === Code 6-4-Setup: Install Qiskit ===

!pip -q install qiskit qiskit-aer pylatexenc

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt

# === Code 6-4: Building a Grover Circuit ===
# Requires window_target from Code 6-0.

# Visualize the measurement results.
def plot_counts(counts, target):
    states = sorted(counts.keys())
    values = [counts[state] for state in states]

    colors = ["gold" if state == target else "silver"
              for state in states]

    plt.figure(figsize=(10.6, 6))

    bars = plt.bar(
        states,
        values,
        color=colors,
        edgecolor="white",
        linewidth=0.4
    )

    bars[states.index(target)].set_hatch("///")

    plt.xlabel("Measured state")
    plt.ylabel("Counts")
    plt.tight_layout()
    plt.show()

# Step 1: Create the circuit and encode the search space.
qc = QuantumCircuit(4, 4)              # 4 search qubits, 4 output bits
target = format(window_target, "04b")  # Binary label used in plots

qc.h(range(4))                         # Equal superposition

# Step 2: Mark the gold patient.
qc.x(0)                                # Align target bit 0
qc.x(2)                                # Align target bit 2
qc.h(3)                                # Prepare phase flip
qc.mcx([0, 1, 2], 3)                   # Mark |1010>
qc.h(3)                                # Complete phase flip
qc.x(0)                                # Restore qubit 0
qc.x(2)                                # Restore qubit 2

# Step 3: Amplify the gold patient.
qc.h(range(4))                         # Change basis
qc.x(range(4))                         # Invert around zero
qc.h(3)                                # Prepare reflection
qc.mcx([0, 1, 2], 3)                   # Reflect about average
qc.h(3)                                # Complete reflection
qc.x(range(4))                         # Undo inversion
qc.h(range(4))                         # Return to search basis

# Step 4: Measure the result.
qc.measure(range(4), range(4))         # Read out all qubits

display(qc.draw("mpl"))

simulator = AerSimulator()
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1024).result()
counts = result.get_counts()

print(f"Gold patient state: |{target}>")
print("Measurement counts:")
print(counts)

plot_counts(counts, target)

### Code 6-5: Estimating Grover at Scale

The examples in this chapter use a small search space so the mechanics remain easy to see. This example steps back and asks a different question: what happens as the search space grows?

The code estimates two quantities for increasingly large searches. The first is the number of qubits required to uniquely identify candidates within the search space. The second is the number of Grover iterations required to locate one of the target states.

The resulting table illustrates how Grover's algorithm scales from a sixteen-candidate visualization window to the full NHANES-inspired search space containing more than eight billion candidate patient-signature pairs. Along the way, the comparison highlights the growing gap between classical search and quantum search as the number of candidates increases.

In [ ]:
import numpy as np
import pandas as pd

# === Code 6-5: Estimating Grover at Full Scale ===
# Requires search_size, needles, and window_size from Code 6-0.

def grover_iterations(candidates, targets=1):
    return int((np.pi / 4) * np.sqrt(candidates / targets))

def qubits_needed(candidates):
    return int(np.ceil(np.log2(candidates)))

scales = [
    (window_size, 1),
    (256, 1),
    (1024, 1),
    (1_000_000, 1),
    (search_size, len(needles))
]

summary = pd.DataFrame([{
    "Search candidates": n,
    "Gold patients": targets,
    "Index qubits": qubits_needed(n),
    "Classical average": n // 2,
    "Grover iterations": grover_iterations(n, targets)
} for n, targets in scales])

display(summary)

print(f"Full search space: {search_size:,} candidates")
print(f"Gold patients: {len(needles)}")
print(f"Estimated Grover iterations: {grover_iterations(search_size, len(needles)):,}")
print(f"Index qubits needed: {qubits_needed(search_size)}")